# VisWord 00 — Setup

One-time bootstrap for all subsequent notebooks:
1. Mount Google Drive (results + cache persist here)
2. Clone the VisWord repo
3. Install missing Python deps (transformers, sentence-transformers, open-clip-torch, pymetric)
4. Vendor the official SALAD repo (one-time git clone)
5. Pin DINOv2 hub commit at the Py3.9-compatible SHA we use on VALAR

**Runtime:** any (CPU fine).  **Wallclock:** ~5 min.

## 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT = '/content/drive/MyDrive/VISWORD'
os.makedirs(PROJECT, exist_ok=True)
os.makedirs(f'{PROJECT}/data', exist_ok=True)
os.makedirs(f'{PROJECT}/runs', exist_ok=True)
os.makedirs(f'{PROJECT}/hf_cache', exist_ok=True)
print('Drive mounted at', PROJECT)

## 2 — Clone repo

Paste your GitHub URL below (or upload the `VISWORD` directory manually to Drive and update the path).

In [ ]:
# Option A: clone from GitHub (if repo is pushed)
REPO_URL = 'https://github.com/hkanpak21/VISWORD.git'  # <-- EDIT if different
REPO_DIR = '/content/VISWORD'
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull
%cd $REPO_DIR
!ls

In [ ]:
# Option B: if repo isn't pushed yet, uncomment and copy from Drive where you uploaded it:
# !cp -r /content/drive/MyDrive/VISWORD_src /content/VISWORD
# %cd /content/VISWORD

## 3 — Install missing Python deps

Colab comes with `torch`, `torchvision`, `datasets`, `PIL`, `matplotlib`, `pydantic` already. We add the ones used by the text-baseline and CLIP rows of the experimental suite.

In [ ]:
!pip install -q transformers sentence-transformers open-clip-torch pytorch-metric-learning huggingface_hub
!pip show transformers | head -3

## 4 — Vendor the SALAD repo

This is the load-bearing dependency that ships inside `third_party/salad/` on VALAR. Same pin, same files.

In [ ]:
SALAD_DIR = '/content/VISWORD/third_party/salad'
if not os.path.exists(SALAD_DIR):
    !mkdir -p /content/VISWORD/third_party
    !git clone https://github.com/serizba/salad.git $SALAD_DIR
    # If SETUP.md pins a specific commit, check it out here:
    # !cd $SALAD_DIR && git checkout <sha_from_SETUP.md>
    !rm -rf $SALAD_DIR/.git
print('salad vendored:', os.path.exists(f'{SALAD_DIR}/models/aggregators/salad.py'))

## 5 — Pin DINOv2 hub cache to Py3.9-compatible commit

Mirrors `scripts/ensure_dinov2_hub.py` from the VALAR project. Avoids PEP-604 `float | None` issues.

In [ ]:
!python /content/VISWORD/scripts/ensure_dinov2_hub.py
# Sanity check:
import torch
m = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
print('DINOv2 loaded:', sum(p.numel() for p in m.parameters())/1e6, 'M params')

## 6 — Point `DATA_DIR` + `HF_HOME` at Drive (persists across sessions)

In [ ]:
os.environ['DATA_DIR'] = f'{PROJECT}/data'
os.environ['HF_HOME'] = f'{PROJECT}/hf_cache'
os.environ['HF_DATASETS_CACHE'] = f'{PROJECT}/hf_cache/datasets'
os.environ['HF_HUB_CACHE'] = f'{PROJECT}/hf_cache/hub'
os.environ['TORCH_HOME'] = f'{PROJECT}/hf_cache/torch'
print('DATA_DIR =', os.environ['DATA_DIR'])
print('HF_HOME =', os.environ['HF_HOME'])

## 7 — Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')
# Prefer A100 for training notebooks; T4 or L4 fine for eval.

## Next step

Proceed to `01_prefetch.ipynb` to download the wiki-ss-corpus (100k–500k rows) to Drive.

Or skip ahead to `02_zeroshot_vision.ipynb` if you've already copied a cache from VALAR to Drive.